# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Authors (IDs): {[author['@id'] for author in metadata.author]}")
print(f"Date published: {metadata.datePublished}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}")

## 2. Data Overview

Review available record sets and their fields using their `@id` values.

Let's inspect which record sets are available in the dataset and enumerate their fields.

In [ ]:
# List all available record sets by @id
print("Available record sets (@id):")
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '')}")
    record_sets_info.append(record_set)

# Show fields for each record set
record_sets_fields = {}
for record_set in record_sets_info:
    fields = record_set.get('field', [])
    # normalize to list of dicts
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = []
    if isinstance(fields, list):
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                field_ids.append(field['@id'])
            elif isinstance(field, str):
                field_ids.append(field)
    record_sets_fields[record_set['@id']] = field_ids
    print(f"\nRecord set {record_set['@id']} fields:")
    for fid in field_ids:
        print(f"    {fid}")

## 3. Data Extraction

Load data from each record set into separate DataFrames. All record sets and fields are referenced by their `@id` identifiers only.

We will create a dictionary of DataFrames for all record sets.

In [ ]:
# Extract data from all record sets using their @id
dataframes = dict()
all_record_set_ids = [record_set['@id'] for record_set in record_sets_info]

for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(dataframes[record_set_id])} records for record set {record_set_id}")
            print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# For demonstration, let's pick the first non-empty record set for analysis
main_record_set_id = None
for k, v in dataframes.items():
    if len(v) > 0:
        main_record_set_id = k
        break

if main_record_set_id is not None:
    print(f"\nSample records from record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No records available for any record set.")

## 4. Exploratory Data Analysis (EDA)

We now apply some analysis to the main record set identified above. We will:
- Select a numeric field (by `@id`) for outlier removal and normalization
- Filter records by a threshold value
- Normalize the numeric field
- Group by a chosen categorical field (`@id`) (if available)

> **Note:** Please inspect the printed DataFrame column names above (these correspond to Croissant field `@id`s) to choose suitable fields for EDA. If the expected field IDs are not present, adjust code according to your real field list.

In [ ]:
# Choose a numeric field and group field by their @ids from the main record set
df = dataframes.get(main_record_set_id)
if df is not None and not df.empty:
    
    # Try to auto-detect numeric fields based on dtype
    numeric_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric field IDs in record set {main_record_set_id}: {numeric_ids}")

    # Select first numeric field
    if numeric_ids:
        numeric_field_id = numeric_ids[0]
        print(f"Using {numeric_field_id} for numeric analysis.")
        
        # Filter records exceeding a threshold (use 10 or median+std for demo)
        try:
            threshold = 10
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())
        except Exception as e:
            print(f"Could not filter on {numeric_field_id}: {e}")
            filtered_df = df.copy()

        # Normalize the numeric field
        if not filtered_df.empty:
            mean = filtered_df[numeric_field_id].mean()
            std = filtered_df[numeric_field_id].std()
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a common non-numeric field
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].nunique() > 1 and not pd.api.types.is_numeric_dtype(df[col])]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping filtered records by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
    else:
        print("No numeric fields detected in this record set.")
else:
    print("No data available for EDA in main record set.")

## 5. Visualization

Visualize data distributions and relationships between fields in the main record set.

We will plot the distribution of the selected numeric field and, if possible, a grouped bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping was performed
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion

In this exploration, we have:
- Loaded and examined the FAIR² clinical dataset and its metadata using `mlcroissant`
- Enumerated all record sets and their fields using `@id`
- Loaded all available record set records into DataFrames
- Performed initial filtering, normalization, and grouping using field `@id`s
- Visualized numeric field distributions and group means

The FAIR² dataset supports secondary analysis of clinicopathological and molecular factors in cancer survivors with second primary colorectal cancer. Adjust groupings, field selections, and thresholds as needed for deeper analytic tasks.